In [1]:
pip install selenium pandas webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [7]:
import json
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

# 1. 셀레니움 드라이버 설정 (헤드리스 모드)
options = webdriver.ChromeOptions()
options.add_argument('--headless') 
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

collected_data = []

try:
    # 2. 다락 지점페이지 접속
    url = "https://www.dalock.kr/service/location"
    print("🔄 다락 사이트 접속 중...")
    driver.get(url)
    time.sleep(4) # 데이터가 로드될 때까지 충분히 대기
    
    print("🔍 페이지 내부 데이터(JSON) 추출 시작...")
    # 3. Next.js가 사용하는 내부 데이터 스크립트 태그 추출
    script_element = driver.find_element(By.ID, "__NEXT_DATA__")
    json_text = script_element.get_attribute("innerHTML")
    
    # 4. JSON 파싱 및 지점 정보 접근
    data = json.loads(json_text)
    
    # 다락의 내부 데이터 구조에 맞춰 지점 리스트 탐색
    # (일반적으로 Next.js의 경우 props.pageProps 내부에 리스트가 존재합니다)
    try:
        # 사이트 구조에 따른 가상 경로 탐색 (실제 구조에 맞춰 자동 예외처리)
        page_props = data.get('props', {}).get('pageProps', {})
        # 지점 목록을 가지고 있는 key 탐색 (예: branches, locations, items 등)
        branches = page_props.get('branches') or page_props.get('branchList') or page_props.get('initialState', {}).get('branches', [])
        
        # 만약 위 경로로 안 찾아질 경우, 전체 JSON에서 'branch' 관련 리스트를 찾는 범용 로직 방어막
        if not branches:
            for key, value in page_props.items():
                if isinstance(value, list) and len(value) > 0 and ('name' in value[0] or 'branchName' in value[0]):
                    branches = value
                    break
    except Exception as e:
        branches = []

    # 5. 만약 JSON 추출이 실패했을 경우를 대비한 2차 백업 (실제 HTML 요소 크롤링)
    if not branches:
        print("⚠️ JSON 추출 실패. HTML 요소를 직접 탐색합니다. (2차 시도)")
        # 다락 사이트의 실제 지점 영역 타겟팅 (정확한 클래스/태그 매칭)
        elements = driver.find_elements(By.CSS_SELECTOR, "div[class*='BranchItem'], [class*='branch_card'], li[class*='item']")
        
        for index, el in enumerate(elements):
            try:
                # 텍스트 추출 후 정제
                text_lines = [line.strip() for line in el.text.split('\n') if line.strip()]
                if len(text_lines) >= 2:
                    branch_name = text_lines[0]
                    address = text_lines[1]
                    region = address.split()[0] if address else "알 수 없음"
                    
                    collected_data.append({
                        "지점명": branch_name,
                        "지점 주소": address,
                        "지역": region,
                        "수집일시": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    })
            except:
                continue
    else:
        # JSON 데이터가 성공적으로 추출되었을 때의 처리
        print(f"✅ 데이터 구조 확인 완료! {len(branches)}개의 지점 정보를 파싱합니다.")
        for branch in branches:
            # 다락 데이터 딕셔너리 키값 매핑 (name, address 등)
            branch_name = branch.get('name') or branch.get('branchName') or branch.get('title')
            address = branch.get('address') or branch.get('roadAddress') or branch.get('addr', '')
            
            if not branch_name:
                continue
                
            region = address.split()[0] if address else "알 수 없음"
            
            collected_data.append({
                "지점명": branch_name,
                "지점 주소": address,
                "지역": region,
                "수집일시": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })

finally:
    driver.quit()

# ==========================================
# 6. 데이터 정제 및 CSV 저장
# ==========================================
if collected_data:
    df = pd.DataFrame(collected_data)
    
    # 혹시 모를 중복 데이터 제거
    df.drop_duplicates(subset=['지점명'], keep='first', inplace=True)
    
    output_filename = "dalock_branches_all.csv"
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n🎉 크롤링 완료! 총 {len(df)}개의 순수 지점 정보가 '{output_filename}' 파일로 저장되었습니다.")
else:
    print("\n❌ 지점 데이터를 수집하지 못했습니다. 다락 사이트의 보안 정책이나 구조가 변경되었는지 확인이 필요합니다.")

🔄 다락 사이트 접속 중...
🔍 페이지 내부 데이터(JSON) 추출 시작...
✅ 데이터 구조 확인 완료! 213개의 지점 정보를 파싱합니다.

🎉 크롤링 완료! 총 213개의 순수 지점 정보가 'dalock_branches_all.csv' 파일로 저장되었습니다.


In [13]:
import json
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

# 1. 셀레니움 드라이버 설정 (헤드리스 모드)
options = webdriver.ChromeOptions()
options.add_argument('--headless') 
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

collected_units = []

try:
    # 2. 다락 지점페이지 접속
    url = "https://www.dalock.kr/service/location"
    print("🔄 [1단계] 다락 사이트 접속 중...")
    driver.get(url)
    time.sleep(5) # 전체 데이터가 내부적으로 로드될 때까지 대기
    
    print("🔍 [2단계] 내부 숨겨진 데이터(JSON) 통째로 추출 시작...")
    # 3. Next.js 백엔드 데이터 스크립트 태그 추출
    script_element = driver.find_element(By.ID, "__NEXT_DATA__")
    json_text = script_element.get_attribute("innerHTML")
    
    # 4. JSON 데이터 파싱
    data = json.loads(json_text)
    page_props = data.get('props', {}).get('pageProps', {})
    
    # 213개 지점이 들어있는 리스트 탐색
    branches = page_props.get('branches') or page_props.get('branchList') or []
    
    if not branches:
        # 혹시 다른 key값에 들어있는지 자동 탐색하는 방어막
        for key, value in page_props.items():
            if isinstance(value, list) and len(value) > 0 and ('name' in value[0] or 'branchName' in value[0]):
                branches = value
                break

    print(f"✅ 총 {len(branches)}개의 지점 데이터 세트를 확인했습니다. 유닛 정보 분해를 시작합니다.")
    print("---------------------------------------------------------------------------")

    # 5. 각 지점 내부에 숨겨진 유닛(Locker) 배열 파싱
    for idx, branch in enumerate(branches):
        branch_name = branch.get('name') or branch.get('branchName') or f"지점_{idx+1}"
        
        # 다락 데이터 내부에서 유닛/가격이 들어있는 key값 매핑 (공통적인 패턴 추적)
        # 보통 상품 정보는 'lockers', 'units', 'products', 'items', 'prices' 등의 이름으로 포함되어 있습니다.
        lockers = branch.get('lockers') or branch.get('units') or branch.get('products') or branch.get('sizes') or []
        
        # 만약 리스트 내부의 또 다른 딕셔너리에 숨어있을 경우를 대비한 범용 탐색
        if not lockers:
            for k, v in branch.items():
                if isinstance(v, list) and len(v) > 0 and any(p in str(v[0]).lower() for p in ['price', 'size', 'name']):
                    lockers = v
                    break
        
        # 유닛 정보가 아예 없는 지점 예외 처리
        if not lockers:
            print(f"[{idx+1}/{len(branches)}] 📍 '{branch_name}' -> 내부 유닛 배열 데이터 없음 (스킵)")
            continue
            
        unit_count = 0
        for locker in lockers:
            # 유닛명(스몰, 미디움 등)과 가격 추출
            unit_type = locker.get('name') or locker.get('size') or locker.get('title') or locker.get('typeName', '기본 유닛')
            
            # 원래 가격 및 할인 가격 가져오기 (숫자형이나 문자형 모두 대응)
            original_price = locker.get('price') or locker.get('originalPrice') or locker.get('normalPrice', '정보 없음')
            discount_price = locker.get('discountPrice') or locker.get('salePrice') or '할인 없음'
            
            # 숫자로 올 경우 보기 좋게 '원' 붙여주기
            if isinstance(original_price, int): original_price = f"{original_price:,}원"
            if isinstance(discount_price, int): discount_price = f"{discount_price:,}원"
            
            collected_units.append({
                "지점명": branch_name,
                "유닛 종류": unit_type,
                "원래 가격": original_price,
                "할인 가격": discount_price,
                "수집일시": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            unit_count += 1
            
        print(f"[{idx+1}/{len(branches)}] 📍 '{branch_name}' -> 유닛 {unit_count}개 파싱 완료")

finally:
    driver.quit()

# ==========================================
# 6. 데이터 정제 및 최종 CSV 저장
# ==========================================
if collected_units:
    df = pd.DataFrame(collected_units)
    
    # 중복 데이터 제거
    df.drop_duplicates(subset=['지점명', '유닛 종류', '원래 가격'], keep='first', inplace=True)
    
    output_filename = "dalock_final_api_units.csv"
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n🎉 [대성공] 누락 없이 전체 데이터 수집 완료!")
    print(f"💾 결과가 '{output_filename}' 파일로 저장되었습니다. (총 {len(df)}건)")
else:
    print("\n❌ 데이터를 수집하지 못했습니다. 다락의 최신 JSON 데이터 구조 분석이 필요합니다.")

🔄 [1단계] 다락 사이트 접속 중...
🔍 [2단계] 내부 숨겨진 데이터(JSON) 통째로 추출 시작...
✅ 총 213개의 지점 데이터 세트를 확인했습니다. 유닛 정보 분해를 시작합니다.
---------------------------------------------------------------------------
[1/213] 📍 '강동성내점' -> 내부 유닛 배열 데이터 없음 (스킵)
[2/213] 📍 '분당수내역점' -> 내부 유닛 배열 데이터 없음 (스킵)
[3/213] 📍 '마천역점' -> 내부 유닛 배열 데이터 없음 (스킵)
[4/213] 📍 '암사역점' -> 내부 유닛 배열 데이터 없음 (스킵)
[5/213] 📍 '수원역점' -> 내부 유닛 배열 데이터 없음 (스킵)
[6/213] 📍 '쌍문점' -> 내부 유닛 배열 데이터 없음 (스킵)
[7/213] 📍 '스페이스마켓1호점' -> 내부 유닛 배열 데이터 없음 (스킵)
[8/213] 📍 '부산동래역점' -> 내부 유닛 배열 데이터 없음 (스킵)
[9/213] 📍 '김포파인스타점' -> 내부 유닛 배열 데이터 없음 (스킵)
[10/213] 📍 '시흥장현점' -> 내부 유닛 배열 데이터 없음 (스킵)
[11/213] 📍 '수원서둔동점' -> 내부 유닛 배열 데이터 없음 (스킵)
[12/213] 📍 '거여초점' -> 내부 유닛 배열 데이터 없음 (스킵)
[13/213] 📍 '창원경상대병원점' -> 내부 유닛 배열 데이터 없음 (스킵)
[14/213] 📍 '속초중앙점' -> 내부 유닛 배열 데이터 없음 (스킵)
[15/213] 📍 '상봉역점' -> 내부 유닛 배열 데이터 없음 (스킵)
[16/213] 📍 '서초방배점' -> 내부 유닛 배열 데이터 없음 (스킵)
[17/213] 📍 '판교점' -> 내부 유닛 배열 데이터 없음 (스킵)
[18/213] 📍 '병점역점' -> 내부 유닛 배열 데이터 없음 (스킵)
[19/213] 📍 '화성봉담점' -> 내부 유닛 배열 데이터 없음 (스킵)
[20/213

In [1]:
import requests
import pandas as pd
import time

LOCATION_URL = "https://www.dalock.kr/_next/data/xbmimhiERlPIwOx02JQF6/ko/service/location.json"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# 지점 목록
data = requests.get(LOCATION_URL, headers=headers).json()

# 여기 구조는 실제 JSON 확인 후 수정 필요
branches = data["pageProps"]["branches"]

rows = []

for branch in branches:

    branch_id = branch["branchId"]
    branch_name = branch["branchName"]

    url = f"https://api.dalock.io/front/v2/branch/{branch_id}/branch-product"

    try:
        products = requests.get(url, headers=headers).json()

        for p in products:
            rows.append({
                "branchId": branch_id,
                "branchName": branch_name,
                "unitType": p.get("unitType"),
                "unitName": p.get("unitName"),
                "sizeInNumber": p.get("sizeInNumber"),
                "price": p.get("price"),
                "discountRate": p.get("discountRate"),
                "discountPrice": p.get("discountPrice")
            })

        print(f"완료: {branch_name}")

    except Exception as e:
        print(f"오류: {branch_name} - {e}")

    time.sleep(0.2)

df = pd.DataFrame(rows)
df.to_csv("dalock_all_units.csv", index=False, encoding="utf-8-sig")

print("저장 완료")

완료: 강동성내점
완료: 분당수내역점
완료: 마천역점
완료: 암사역점
완료: 수원역점
완료: 쌍문점
완료: 스페이스마켓1호점
완료: 부산동래역점
완료: 김포파인스타점
완료: 시흥장현점
완료: 수원서둔동점
완료: 거여초점
완료: 창원경상대병원점
완료: 속초중앙점
완료: 상봉역점
완료: 서초방배점
완료: 판교점
완료: 병점역점
완료: 화성봉담점
완료: 용인동백점
완료: 인천구월동점
완료: 해운대장산역점
완료: 삼송역점
완료: 선정릉역점
완료: 인덕원점
완료: 새절역점
완료: 덕성여대점
완료: 창동역점
완료: 망포점
완료: 고덕역점
완료: 낙성대역점
완료: 운서역점
완료: 외대점
완료: 동대문리마크빌
완료: 중탑사거리점
완료: 합정역점
완료: 향동점
완료: 부평삼산점
완료: 송도센텀점
완료: 김포마산점
완료: 상동역점
완료: 보문역점
완료: 동탄호수공원점
완료: 송도테크노파크점
완료: 종각역점
완료: 이수사당점
완료: 광교역점
완료: 김포풍무점
완료: 노량진역점
완료: 수원경희대점
완료: 부평구청역점
완료: 청량리역점
완료: 수원영통점
완료: 올림픽공원점
완료: 암사동점
완료: 서울대벤처타운역점
완료: 독산현대지식산업센터점
완료: 송내역점
완료: 부산연산점
완료: 염창로보점
완료: 평택캠프험프리스점
완료: 여수시청점
완료: 용인죽전점
완료: 수유2호점
완료: 수원곡반정점
완료: 연수점
완료: 이랜드 PEER 대명
완료: 세마역점
완료: 대전용문역점
완료: 이랜드 PEER 둔산
완료: 철산점
완료: 부천역점
완료: 청주복대점
완료: 안양만안점
완료: 면목역점
완료: 구로하이엔드점
완료: 오금역점
완료: 신논현역점
완료: 황학아크로타워점
완료: DMC역공항철도점
완료: 김포양촌점
완료: 도림천역점
완료: 기흥서천점
완료: 중랑역점
완료: 동탄IX타워점
완료: 성남은행동점
완료: 송도3호점
완료: 가산디지털단지역점
완료: 이수역사거리점
완료: 구로STX-W타워점
완료: 성산점
완료: 청라점
완료: 시흥은계점
완료: 군자역7번출구점
완료: 삼성역점
완료: 동수원사거리점
완